# YOLO CIFAR10: Fine-Tuned Teacher → Pruned Student → Logits Distillation (Classification)

This notebook runs a YOLOv8n-cls **classification** pruning + logits distillation flow on CIFAR10:
1. Load a **CIFAR10-fine-tuned** YOLOv8n-cls teacher (10-class head).
2. Build a `MaseGraph` from the teacher weights and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images + labels).
4. Evaluate teacher, pruned-only (no KD) student, and distilled student on CIFAR10 validation set.

In [1]:
cls_model_checkpoint = "data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt"
cls_device = "cuda"
cifar_root = "./data"

image_size = 32
cifar_batch_size = 16

cifar_prune_sparsity = 0.3
cifar_kd_alpha = 0.8
cifar_kd_temperature = 2.0
cifar_kd_epochs = 1
lr = 1e-4
seed = 42


## Imports and setup

In [2]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolo

from mase_kd.core.losses import DistillationLossConfig, hard_label_ce_loss, soft_logit_kl_loss
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")


/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [3]:
if cls_device == "cuda" and not torch.cuda.is_available():
    cls_device = "cpu"

torch.manual_seed(seed)
random.seed(seed)

print(f"Using device: {cls_device}")

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

cifar_transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_dataset = datasets.CIFAR10(root=cifar_root, train=True, transform=cifar_transform_train, download=True)
val_dataset = datasets.CIFAR10(root=cifar_root, train=False, transform=cifar_transform_eval, download=True)

# drop_last=True keeps batch size fixed for the FX-specialized pruned student graph.
train_loader = DataLoader(
    train_dataset,
    batch_size=cifar_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cifar_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


Train: 50000 | Val: 10000


## Load CIFAR10-fine-tuned teacher (YOLOv8n-cls, 10 classes)

In [5]:
ultra_teacher = YOLO(cls_model_checkpoint)
teacher_cls_model = ultra_teacher.model.to(cls_device)
teacher_cls_model.eval()

nc = ultra_teacher.model.yaml.get("nc", 10)
print(f"Loaded CIFAR10-fine-tuned teacher: {cls_model_checkpoint}")
print(f"Teacher num_classes (nc): {nc}")

Loaded CIFAR10-fine-tuned teacher: data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt
Teacher num_classes (nc): 10


## Build MASE graph from teacher weights and prune to create student

In [6]:
trace_input_cls = torch.randn(cifar_batch_size, 3, image_size, image_size)

# Build a MaseYoloClassificationModel with the CIFAR10 class count, then load
# the fine-tuned teacher weights. We bypass get_yolo_classification_model()
# because it expects the original "yolov8n-cls.pt" name and 1000-class arch.
student_seed_cls_model = MaseYoloClassificationModel(cfg="yolov8n-cls.yaml", nc=nc)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.load_state_dict(ultra_teacher.model.state_dict())
student_seed_cls_model.train()

mg_cls = MaseGraph(student_seed_cls_model, cf_args={"x": trace_input_cls})
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
        "add_value": True,
    },
)

pruning_config_cls = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
}

mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)
student_cls_model = mg_cls.model.to(cls_device)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()

print(f"Pruning complete ({cifar_prune_sparsity*100:.0f}% sparsity); using pruned teacher as student.")

Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             


  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    343050  ultralytics.nn.modules.head.Classify         [256, 10]                     
YOLOv8n-cls summary: 56 layers, 1,451,098 parameters, 1,451,098 gradients, 3.4 GFLOPs
tensor([[[[-4.0005e-03]],

         [[-1.5646e-01]],

         [[-2.3093e-04]],

         ...,

         [[-6.5909e-07]],

         [[-2.4836e-09]],

         [[-2.4167e-02]]],


        [[[-1.5091e-02]],

         [[-1.1201e-02]],

         [[-2.1692e-04]],

         ...,

         [[-1.6136e-06]],

         [[-1.4263e-03]],

         [[-8.2

/usr/local/lib/python3.11/dist-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(
INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: m

Pruning complete (30% sparsity); using pruned teacher as student.


## Distill teacher into pruned student on CIFAR10 images + labels

In [ ]:
kd_config_cls = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)

optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=lr)

distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=cls_device,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_train_epochs=cifar_kd_epochs,
    eval_teacher=True,
)

loss_history = distiller_cls.train()


Epoch 1/1
  Batch 0001/3125 | total=5.146288 | hard=4.684159 | soft=5.261820
  Batch 0010/3125 | total=4.523819 | hard=3.432981 | soft=4.796528
  Batch 0020/3125 | total=3.021136 | hard=2.555301 | soft=3.137595
  Batch 0030/3125 | total=3.201406 | hard=2.942308 | soft=3.266181
  Batch 0040/3125 | total=2.777248 | hard=2.317704 | soft=2.892134
  Batch 0050/3125 | total=2.844021 | hard=2.532936 | soft=2.921792
  Batch 0060/3125 | total=2.622995 | hard=2.449339 | soft=2.666409
  Batch 0070/3125 | total=2.823237 | hard=2.567487 | soft=2.887174
  Batch 0080/3125 | total=2.915195 | hard=2.387143 | soft=3.047208
  Batch 0090/3125 | total=2.772204 | hard=2.396124 | soft=2.866224
  Batch 0100/3125 | total=2.499079 | hard=2.384252 | soft=2.527786
  Batch 0110/3125 | total=2.180504 | hard=2.478136 | soft=2.106096
  Batch 0120/3125 | total=3.090605 | hard=2.431444 | soft=3.255395
  Batch 0130/3125 | total=2.959193 | hard=2.561233 | soft=3.058684
  Batch 0140/3125 | total=2.761896 | hard=2.370528 |

## Evaluation: teacher vs pruned-only vs distilled student (CIFAR10 validation)

In [8]:
import time

# evaluate_model_on_cifar10_val is kept as a standalone helper for the
# pruned-no-KD baseline, which is not managed by the distiller.
@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device):
    """Evaluate a model on the full validation loader. Returns top-1 accuracy, CE loss, and timing."""
    model.eval()
    batches = 0
    samples = 0
    total_forward_ms = 0.0
    correct_top1 = 0
    total_ce_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            total_ce_loss += hard_label_ce_loss(logits, labels).item()

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "top1_acc": correct_top1 / max(samples, 1),
        "avg_ce_loss": total_ce_loss / max(batches, 1),
    }

# --- Training loss summary ---
if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD training batches")
    print(f"  First loss: {loss_history[0]:.6f}")
    print(f"  Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

# --- Evaluate teacher + distilled student via the distiller ---
print("\n--- CIFAR10 Validation Results ---\n")

eval_results = distiller_cls.evaluate()
teacher_metrics = eval_results.get("teacher")
student_metrics = eval_results["student"]
val_kd_loss = eval_results["val_kd_loss"]
kd_batches = eval_results["kd_batches"]

# --- Pruned-no-KD baseline (evaluated standalone, not via the distiller) ---
pruned_no_kd_metrics = None
if "pruned_no_kd_model" in globals():
    pruned_no_kd_metrics = evaluate_model_on_cifar10_val(pruned_no_kd_model, val_loader, cls_device)

# --- Print results ---
if teacher_metrics is not None:
    print(f"Teacher (fine-tuned on CIFAR10):")
    print(
        f"  top1_acc={teacher_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={teacher_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={teacher_metrics['samples']}"
    )

if pruned_no_kd_metrics is not None:
    print(f"\nPruned student (NO distillation, {cifar_prune_sparsity*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_no_kd_metrics['samples']}"
    )

print(f"\nDistilled student (pruned + {cifar_kd_epochs} epochs, alpha={cifar_kd_alpha}, T={cifar_kd_temperature}):")
print(
    f"  top1_acc={student_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={student_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={student_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={student_metrics['samples']}"
)

print(f"\nValidation KD loss (teacher vs distilled student, T={cifar_kd_temperature}): "
      f"{val_kd_loss:.6f} over {kd_batches} batches")

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Model':<40} {'Top-1 Acc':>10} {'CE Loss':>10}")
print(f"{'─'*40} {'─'*10} {'─'*10}")
if teacher_metrics is not None:
    print(f"{'Teacher (fine-tuned)':<40} {teacher_metrics['top1_acc']*100:>9.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")
if pruned_no_kd_metrics is not None:
    print(f"{'Pruned student (no KD)':<40} {pruned_no_kd_metrics['top1_acc']*100:>9.2f}% {pruned_no_kd_metrics['avg_ce_loss']:>10.4f}")
print(f"{'Distilled student (pruned + KD)':<40} {student_metrics['top1_acc']*100:>9.2f}% {student_metrics['avg_ce_loss']:>10.4f}")


Recorded 3125 KD training batches
  First loss: 5.146288
  Last loss:  2.748096

--- CIFAR10 Validation Results ---



Teacher (fine-tuned on CIFAR10):
  top1_acc=75.09% | CE_loss=1.8045 | fwd_ms/batch=4.79 | samples=10000

Pruned student (NO distillation, 30% sparsity):
  top1_acc=9.82% | CE_loss=2.6137 | fwd_ms/batch=16.86 | samples=10000

Distilled student (pruned + 1 epochs, alpha=0.8, T=2.0):
  top1_acc=9.61% | CE_loss=2.3122 | fwd_ms/batch=16.34 | samples=10000

Validation KD loss (teacher vs distilled student, T=2.0): 0.040221 over 625 batches

--- Summary ---
Model                                     Top-1 Acc    CE Loss
──────────────────────────────────────── ────────── ──────────
Teacher (fine-tuned)                         75.09%     1.8045
Pruned student (no KD)                        9.82%     2.6137
Distilled student (pruned + KD)               9.61%     2.3122
